# 🧪 Lab Report Explainer: accuracy evaluation

This notebook measures how accurate the app is, using **synthetic reports with known correct answers**:

1. It generates fake lab reports and fake cancer biopsy reports (no real patient data).
2. It sends each one through the **real app code** and Gemini.
3. It scores the answers, comparing **AI alone** against **AI + the app's code checks**.

**Before you start**
- Add your Gemini key to Colab: click the **🔑 key icon** in the left sidebar → **Add new secret** → name it `GEMINI_API_KEY`, paste your key, and turn on **Notebook access**.
- Then click **Runtime → Run all**. With the default settings (32 reports) it takes about **5–10 minutes** and uses about 32 Gemini requests.
- If the free quota runs out, just run the notebook again later: it continues where it stopped.

In [ ]:
# Setup: download the project and install it (about 1 minute in Colab)
import os, sys, pathlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("/content/lab-report-explainer").exists():
        !git clone -q https://github.com/srinathgupta/lab-report-explainer.git /content/lab-report-explainer
    %cd /content/lab-report-explainer
    !pip install -q -r requirements.txt matplotlib tqdm
elif pathlib.Path.cwd().name == "eval":
    os.chdir("..")  # running locally from the eval/ folder
sys.path.insert(0, os.getcwd())
print("Ready in", os.getcwd())

## Settings
Change these if you like. Keep `DRY_RUN = False` for real scores. `DRY_RUN = True` uses a fake Gemini that makes mistakes on purpose, to test the notebook without a key; **its scores are not real**.

In [ ]:
from medexplain.core import DEFAULT_MODEL

DRY_RUN = False        # True = fake Gemini, no key needed, scores are NOT real
N_LAB = 20             # synthetic lab reports
N_ONCO = 12            # synthetic cancer biopsy reports (breast, colon, lung)
MODEL = DEFAULT_MODEL  # the same model the app uses
MODE = "fast"          # the app's default; try "accurate" to compare
PACE_SECONDS = 6       # wait between requests, to stay inside the free-tier rate limit
RUN_HINDI = False      # also run a few lab reports with Hindi explanations
N_HINDI = 5
print("Model:", MODEL, "| Dry run:", DRY_RUN)

In [ ]:
# Gemini key (from Colab secrets, or typed in privately)
if not DRY_RUN and not os.environ.get("GEMINI_API_KEY"):
    key = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception as e:
            print("Couldn't read the Colab secret:", e)
    if not key:
        from getpass import getpass
        key = getpass("Paste your Gemini API key (it won't be shown): ")
    os.environ["GEMINI_API_KEY"] = key.strip()
print("Key set" if os.environ.get("GEMINI_API_KEY") or DRY_RUN else "No key!")

## 1. Generate the synthetic reports

In [ ]:
import pandas as pd
from pathlib import Path
from eval import evalset, run_eval

DATA = Path("eval_data")
cases = evalset.build(DATA, N_LAB, N_ONCO)
lab_results = [t for c in cases for t in c.lab_tests]
print(f"{len(cases)} reports: {N_LAB} lab ({len(lab_results)} test results), {N_ONCO} cancer biopsy")
print("Correct answers:", pd.Series([t.status for t in lab_results]).value_counts().to_dict())
pd.DataFrame([t.__dict__ for t in cases[0].lab_tests]).head(8)

## 2. Run every report through the app

In [ ]:
from tqdm.auto import tqdm

if DRY_RUN:
    from eval.fake_gemini import FakeGemini
    client = FakeGemini(cases, DATA)
    RUN_MODEL = "dry-run"  # kept separate from real results
else:
    from medexplain.core import make_client
    client = make_client()
    RUN_MODEL = MODEL

results = run_eval.run_all(cases, DATA, client, RUN_MODEL, "English", mode=MODE,
                           pace_s=0 if DRY_RUN else PACE_SECONDS, progress=tqdm)
print(f"Finished {len(results)} of {len(cases)} reports")

## 3. Scores

In [ ]:
scores = run_eval.score(results)
if DRY_RUN:
    print("⚠️ DRY RUN: these scores come from a fake Gemini and are NOT real.")
table = pd.DataFrame([(label, f"{scores[k]}%") for k, label in run_eval.LABELS.items() if scores.get(k) is not None],
                     columns=["Check", "Score"])
print(f"Errors: {scores['errors']} · Median time per report: {scores['median_seconds']} s")
print(f"Code check fixed {scores['code_check_fixed']} AI mistakes and introduced {scores['code_check_broke']}")
table

In [ ]:
import matplotlib.pyplot as plt

pairs = [("Normal/high/low\ncorrect", "status_ai_only_pct", "status_with_code_check_pct"),
         ("Abnormal results\ncaught", "abnormal_caught_ai_only_pct", "abnormal_caught_with_code_check_pct")]
fig, ax = plt.subplots(figsize=(7, 3.6))
x = range(len(pairs))
ai = [scores[a] or 0 for _, a, _ in pairs]
both = [scores[b] or 0 for _, _, b in pairs]
ax.bar([i - 0.2 for i in x], ai, width=0.4, label="AI alone", color="#9ecae1")
ax.bar([i + 0.2 for i in x], both, width=0.4, label="AI + code check", color="#08519c")
for i, (a, b) in enumerate(zip(ai, both)):
    ax.text(i - 0.2, a + 1, f"{a:.0f}%", ha="center"); ax.text(i + 0.2, b + 1, f"{b:.0f}%", ha="center")
ax.set_xticks(list(x), [p[0] for p in pairs]); ax.set_ylim(0, 110); ax.set_ylabel("%")
ax.set_title("What the app's code check adds" + (" (DRY RUN, not real)" if DRY_RUN else ""))
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

## 4. What it still gets wrong
Looking at the mistakes is the most useful part: each row is a result the app got wrong, even after the code check.

In [ ]:
mistakes = pd.DataFrame(run_eval.mistakes(results))
print(f"{len(mistakes)} lab results still wrong")
mistakes.head(20) if len(mistakes) else "No mistakes 🎉"

In [ ]:
onco = pd.DataFrame([r for r in results if r.get("kind") == "onco" and "error" not in r])
if len(onco):
    display(onco[["case_id", "site", "truth_tnm", "ai_tnm", "tnm_ok", "truth_grade", "ai_grade", "grade_ok",
                  "markers_found", "markers_total", "prognosis_in_ai", "prognosis_after_filter"]])

## 5. Optional: Hindi explanations
Checks that values stay exactly as printed when the explanation language changes. Set `RUN_HINDI = True` in Settings first.

In [ ]:
if RUN_HINDI:
    lab_cases = [c for c in cases if c.kind == "lab"][:N_HINDI]
    hindi = run_eval.run_all(lab_cases, DATA, client, RUN_MODEL, "Hindi", mode=MODE,
                             pace_s=0 if DRY_RUN else PACE_SECONDS, progress=tqdm)
    h = run_eval.score(hindi)
    print(f"Hindi: values exact {h['values_exact_pct']}% · status with code check {h['status_with_code_check_pct']}%")
else:
    print("Skipped (RUN_HINDI = False)")

## 6. Save the results
Copy the table below into your README. The results file can be downloaded to keep a record.

In [ ]:
snippet = run_eval.readme_table(scores, MODEL)
if DRY_RUN:
    snippet = "> ⚠️ DRY RUN: fake scores, do not publish.\n\n" + snippet
print(snippet)
(DATA / "README_snippet.md").write_text(snippet)
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("eval_results", "zip", DATA, ".")
    files.download("eval_results.zip")